# Rebuild training charts from logs / CSVs (no Weights & Biases)

Reconstructs the **loss / pixel-accuracy / mIoU / per-class-IoU** charts for a binary
`swin_binary_segmentation_221.py` run straight from the data it wrote to disk — so you
don't need W&B at all.

**Two supported inputs (per run):**
- a run **`.log`** file (e.g. `binary_run_focal_cw.log`, `binary_run_sched.log`) — the cleanest
  per-run source, since different runs wrote different log files, and
- the per-fold **`val_metrics.csv`** files.

**Getting the files in:** run the upload cell and drag in one or more `.log` files from your
Lightning Studio (download them from the Studio file browser first), **or** point at a Drive
path if you synced the outputs there.


## 0. Imports

In [ ]:
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

## 1. Get your file(s) into Colab

**Option A — upload** (drag in `binary_run_*.log` and/or `val_metrics.csv`):

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick one or more .log / .csv files
paths = list(uploaded.keys())      # files.upload() saves them into the Colab working dir
print('loaded:', paths)

**Option B — from Google Drive** (skip Option A if you use this). Uncomment, mount, and list
the run logs you synced to Drive:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/My Drive/Petrographic images_ML work/model_outputs_lightning/seg_with_aug_binary_3fold'
# paths = [f'{BASE}/binary_run_sched.log', f'{BASE}/binary_run_focal_cw.log']
# print('loaded:', paths)

## 2. Parsers (log + csv -> tidy DataFrame)

In [ ]:
EPOCH_RE = re.compile(
    r"Fold\s+(\d+)\s*\|\s*Epoch\s+(\d+)\s*\|\s*train\s+([\d.]+)\s*\|\s*val\s+([\d.]+)"
    r"\s*\|\s*acc\s+([\d.]+)\s*\|\s*mIoU\s+([\d.]+)")
BG_IOU_RE   = re.compile(r"background IoU:\s*([\d.]+|nan)")
GRAIN_IOU_RE= re.compile(r"grain IoU:\s*([\d.]+|nan)")
PRED_RE     = re.compile(r"prediction\s*->\s*background:\s*([\d.]+)%\s*\|\s*grain:\s*([\d.]+)%")
COLS = ['fold','epoch','train_loss','val_loss','pixel_acc','miou',
        'iou_bg','iou_grain','pred_bg_pct','pred_grain_pct']

def _f(x):
    try: return float(x)
    except (TypeError, ValueError): return float('nan')

def parse_log(path):
    rows, cur = [], None
    for line in Path(path).read_text(errors='ignore').splitlines():
        m = EPOCH_RE.search(line)
        if m:
            if cur is not None: rows.append(cur)
            cur = {'fold':int(m.group(1)), 'epoch':int(m.group(2)),
                   'train_loss':_f(m.group(3)), 'val_loss':_f(m.group(4)),
                   'pixel_acc':_f(m.group(5)), 'miou':_f(m.group(6)),
                   'iou_bg':float('nan'), 'iou_grain':float('nan'),
                   'pred_bg_pct':float('nan'), 'pred_grain_pct':float('nan')}
            continue
        if cur is None: continue
        m = BG_IOU_RE.search(line)
        if m: cur['iou_bg'] = _f(m.group(1)); continue
        m = GRAIN_IOU_RE.search(line)
        if m: cur['iou_grain'] = _f(m.group(1)); continue
        m = PRED_RE.search(line)
        if m: cur['pred_bg_pct'], cur['pred_grain_pct'] = _f(m.group(1)), _f(m.group(2))
    if cur is not None: rows.append(cur)
    return pd.DataFrame(rows, columns=COLS)

def parse_csv(path):
    df = pd.read_csv(path)
    if 'fold' not in df.columns:            # a single fold_*/val_metrics.csv has no fold column
        df['fold'] = 0
    for c in COLS:
        if c not in df.columns: df[c] = float('nan')
    return df[COLS]

def load_any(path):
    return parse_csv(path) if str(path).lower().endswith('.csv') else parse_log(path)

## 3. Plot one run (2x2: loss, accuracy, mIoU, per-class IoU) — one line per fold

In [ ]:
def plot_run(df, title):
    folds = sorted(df['fold'].unique())
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(title, fontsize=13)
    panels = [
        (axes[0,0], 'Loss',            [('train_loss','train'), ('val_loss','val')]),
        (axes[0,1], 'Pixel accuracy',  [('pixel_acc','acc')]),
        (axes[1,0], 'mIoU',            [('miou','mIoU')]),
        (axes[1,1], 'Per-class val IoU',[('iou_bg','background'), ('iou_grain','grain')]),
    ]
    for ax, ttl, keys in panels:
        for fold in folds:
            sub = df[df['fold'] == fold].sort_values('epoch')
            for key, lbl in keys:
                ax.plot(sub['epoch'], sub[key], marker='.', ms=3, label=f'fold{fold} {lbl}')
        ax.set_title(ttl); ax.set_xlabel('epoch'); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
    axes[0,1].set_ylim(0, 1); axes[1,0].set_ylim(0, 1); axes[1,1].set_ylim(-0.02, 1)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()
    return fig

## 4. Run it — parse + chart every file you loaded

In [ ]:
dfs = {}
for p in paths:
    name = Path(p).name
    df = load_any(p)
    if df.empty:
        print(f'!! no epoch records found in {name} — wrong file?'); continue
    dfs[name] = df
    print(f'\n=== {name}: {len(df)} epoch-rows, folds {sorted(df.fold.unique())} ===')
    display(df.head())
    fig = plot_run(df, name)
    fig.savefig(name + '_charts.png', dpi=150, bbox_inches='tight')   # also saved to Colab dir
    df.to_csv(name + '_metrics.csv', index=False)

## 5. (Optional) Compare runs — best val mIoU per fold

Handy for CE vs focal vs focal_cw vs sched. Uses each run's best epoch per fold.

In [ ]:
if len(dfs) > 1:
    summary = []
    for name, df in dfs.items():
        for fold, sub in df.groupby('fold'):
            best = sub.loc[sub['miou'].idxmax()]
            summary.append({'run': name.replace('.log','').replace('binary_run_',''),
                            'fold': int(fold), 'best_epoch': int(best['epoch']),
                            'best_miou': round(best['miou'], 4),
                            'bg_iou': round(best['iou_bg'], 4),
                            'grain_iou': round(best['iou_grain'], 4)})
    comp = pd.DataFrame(summary)
    display(comp)
    piv = comp.pivot_table(index='run', values='best_miou', aggfunc='mean').sort_values('best_miou')
    print('\nMean best mIoU across folds, per run:'); display(piv)
else:
    print('Load 2+ runs to compare.')

## 6. (Optional) Download the charts/CSVs to your computer

In [ ]:
from google.colab import files
import glob
for f in glob.glob('*_charts.png') + glob.glob('*_metrics.csv'):
    files.download(f)